In [4]:
"""
Temperature Forecast Data Preparation Stub

- Target: T_anom(t + lead)
- Features:
    * Lagged T_anom
    * Spatial features (lat, lon, elevation)
    * Seasonal phase (sin_month, cos_month)
- Train climatology computed using training window only
- No modeling performed
"""

import sqlite3
import pandas as pd
import numpy as np


# ============================================================
# CONFIGURATION
# ============================================================

SQLITE_DB = "/home/joe/work/Fire/Data/DB/era5DataMeans.db"
# DATA_TABLE = "MEANS_TTdRHVPD"
POINT_META_CSV = "Data/latlon_elevation.csv"

TARGET_VAR = "T"

# --- climatology window (TRAIN ONLY) ---
TRAIN_CLIMO_START_YM = 1990 * 12 + 1
TRAIN_CLIMO_END_YM   = 2015 * 12 + 12

# --- forecast leads ---
LEADS = list(range(1, 13))  # 1–12 month leads

# --- lag configuration ---
LAG_MONTHS = [0, 1, 2, 3]   # configurable

# --- train/test split ---
TEST_START_YM = 2016 * 12 + 1

# --- spatial selection (optional) ---
CENTER_POINT_ID = 141
GRID_RADIUS_DEG = 0.2


# ============================================================
# DATA LOADING
# ============================================================

def load_data():
    conn = sqlite3.connect(SQLITE_DB)
    df = pd.read_sql(f"SELECT * FROM MEANS_TTdRHVPD", conn)
    conn.close()

    df = df.rename(columns={"Point": "point_id"})
    df["Yrmo"] = df["Yrmo"].astype(int)
    df["year"] = df["Yrmo"] // 100
    df["month"] = df["Yrmo"] % 100
    df["ym_index"] = df["year"] * 12 + df["month"]

    return df


def load_metadata():
    meta = pd.read_csv(POINT_META_CSV)
    meta = meta.rename(columns={
        "point": "point_id",
        "latitude": "lat",
        "longitude": "lon",
        "elevation_m": "elevation"
    })
    return meta


# ============================================================
# FEATURE ENGINEERING
# ============================================================

def apply_spatial_subset(df, meta):
    df = df.merge(meta, on="point_id", how="left")

    if CENTER_POINT_ID is not None:
        center = meta.loc[meta.point_id == CENTER_POINT_ID].iloc[0]
        df = df[
            (np.abs(df.lat - center.lat) <= GRID_RADIUS_DEG) &
            (np.abs(df.lon - center.lon) <= GRID_RADIUS_DEG)
        ].copy()

    return df


def compute_climatology(df):
    mask = (
        (df.ym_index >= TRAIN_CLIMO_START_YM) &
        (df.ym_index <= TRAIN_CLIMO_END_YM)
    )

    climo = (
        df.loc[mask]
          .groupby(["point_id", "month"], as_index=False)[TARGET_VAR]
          .mean()
          .rename(columns={TARGET_VAR: "T_climo"})
    )

    return climo


def apply_anomalies(df, climo):
    df = df.merge(climo, on=["point_id", "month"], how="left")
    df["T_anom"] = df[TARGET_VAR] - df["T_climo"]
    return df


def add_seasonal_features(df):
    df["sin_month"] = np.sin(2 * np.pi * df["month"] / 12)
    df["cos_month"] = np.cos(2 * np.pi * df["month"] / 12)
    return df


def add_lagged_features(df):
    df = df.sort_values(["point_id", "ym_index"])

    for lag in LAG_MONTHS:
        df[f"T_anom_lag{lag}"] = (
            df.groupby("point_id")["T_anom"].shift(lag)
        )

    return df


# ============================================================
# BUILD FORECAST TABLE
# ============================================================

def build_forecast_table(df):
    rows = []

    for pid, df_p in df.groupby("point_id"):
        df_p = df_p.reset_index(drop=True)

        for i, issue in df_p.iterrows():
            for lead in LEADS:
                tgt_idx = i + lead
                if tgt_idx >= len(df_p):
                    continue

                target = df_p.loc[tgt_idx]

                row = {
                    "point_id": pid,
                    "issue_ym": issue.ym_index,
                    "lead": lead,
                    "target_ym": target.ym_index,
                    "y": target.T_anom,
                    "lat": issue.lat,
                    "lon": issue.lon,
                    "elevation": issue.elevation,
                    "sin_month": issue.sin_month,
                    "cos_month": issue.cos_month
                }

                for lag in LAG_MONTHS:
                    row[f"T_anom_lag{lag}"] = issue[f"T_anom_lag{lag}"]

                rows.append(row)

    forecast_df = pd.DataFrame(rows)

    lag_cols = [f"T_anom_lag{lag}" for lag in LAG_MONTHS]
    forecast_df = forecast_df.dropna(subset=lag_cols).reset_index(drop=True)

    return forecast_df


# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

def train_test_split(forecast_df):
    train_df = forecast_df[forecast_df.issue_ym < TEST_START_YM].copy()
    test_df  = forecast_df[forecast_df.issue_ym >= TEST_START_YM].copy()

    feature_cols = (
        [f"T_anom_lag{lag}" for lag in LAG_MONTHS] +
        ["lat", "lon", "elevation", "sin_month", "cos_month"]
    )

    X_train = train_df[feature_cols].copy()
    y_train = train_df["y"].copy()

    X_test  = test_df[feature_cols].copy()
    y_test  = test_df["y"].copy()

    return X_train, X_test, y_train, y_test, train_df, test_df


# ============================================================
# MASTER PIPELINE
# ============================================================

def prepare_temperature_dataset():
    df = load_data()
    meta = load_metadata()

    df = apply_spatial_subset(df, meta)
    climo = compute_climatology(df)
    df = apply_anomalies(df, climo)
    df = add_seasonal_features(df)
    df = add_lagged_features(df)

    forecast_df = build_forecast_table(df)

    return train_test_split(forecast_df)


# ============================================================
# RUN PIPELINE
# ============================================================

if __name__ == "__main__":
    X_train, X_test, y_train, y_test, train_df, test_df = prepare_temperature_dataset()

    print("X_train shape:", X_train.shape)
    print("X_test shape :", X_test.shape)
    print("Number of features:", X_train.shape[1])


X_train shape: (33372, 9)
X_test shape : (10962, 9)
Number of features: 9


In [5]:
train_df

,point_id,issue_ym,lead,target_ym,y,lat,lon,elevation,sin_month,cos_month,T_anom_lag0,T_anom_lag1,T_anom_lag2,T_anom_lag3
0,68,23884.0,1,23885.0,-1.634418,41.1,257.8,1100.661075,8.660254e-01,-0.5,-0.140510,-0.607937,-0.840505,1.350123
1,68,23884.0,2,23886.0,1.798526,41.1,257.8,1100.661075,8.660254e-01,-0.5,-0.140510,-0.607937,-0.840505,1.350123
2,68,23884.0,3,23887.0,-1.046656,41.1,257.8,1100.661075,8.660254e-01,-0.5,-0.140510,-0.607937,-0.840505,1.350123
3,68,23884.0,4,23888.0,0.061669,41.1,257.8,1100.661075,8.660254e-01,-0.5,-0.140510,-0.607937,-0.840505,1.350123
4,68,23884.0,5,23889.0,1.869332,41.1,257.8,1100.661075,8.660254e-01,-0.5,-0.140510,-0.607937,-0.840505,1.350123
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43111,212,24192.0,8,24200.0,-0.792838,40.9,258.0,1090.418010,-2.449294e-16,1.0,-0.282395,0.671766,2.540248,2.881024
43112,212,24192.0,9,24201.0,0.856107,40.9,258.0,1090.418010,-2.449294e-16,1.0,-0.282395,0.671766,2.540248,2.881024
43113,212,24192.0,10,24202.0,3.271740,40.9,258.0,1090.418010,-2.449294e-16,1.0,-0.282395,0.671766,2.540248,2.881024
43114,212,24192.0,11,24203.0,3.991974,40.9,258.0,1090.418010,-2.449294e-16,1.0,-0.282395,0.671766,2.540248,2.881024


In [6]:
X_train

,T_anom_lag0,T_anom_lag1,T_anom_lag2,T_anom_lag3,lat,lon,elevation,sin_month,cos_month
0,-0.140510,-0.607937,-0.840505,1.350123,41.1,257.8,1100.661075,8.660254e-01,-0.5
1,-0.140510,-0.607937,-0.840505,1.350123,41.1,257.8,1100.661075,8.660254e-01,-0.5
2,-0.140510,-0.607937,-0.840505,1.350123,41.1,257.8,1100.661075,8.660254e-01,-0.5
3,-0.140510,-0.607937,-0.840505,1.350123,41.1,257.8,1100.661075,8.660254e-01,-0.5
4,-0.140510,-0.607937,-0.840505,1.350123,41.1,257.8,1100.661075,8.660254e-01,-0.5
...,...,...,...,...,...,...,...,...,...
43111,-0.282395,0.671766,2.540248,2.881024,40.9,258.0,1090.418010,-2.449294e-16,1.0
43112,-0.282395,0.671766,2.540248,2.881024,40.9,258.0,1090.418010,-2.449294e-16,1.0
43113,-0.282395,0.671766,2.540248,2.881024,40.9,258.0,1090.418010,-2.449294e-16,1.0
43114,-0.282395,0.671766,2.540248,2.881024,40.9,258.0,1090.418010,-2.449294e-16,1.0


In [7]:
X_train.columns

Index(['T_anom_lag0', 'T_anom_lag1', 'T_anom_lag2', 'T_anom_lag3', 'lat',
       'lon', 'elevation', 'sin_month', 'cos_month'],
      dtype='object')